# Lahore UC

## Import Libraries and initialisation

In [1]:
import ee, geemap, geopandas as gpd, pandas as pd
import folium

ee.Authenticate()
ee.Initialize()

/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Read shape files

In [3]:

gdf = gpd.read_file("../../data/Lahore UCs/Lahore UC.shp")

# Optional: restrict to Lahore district only if present
if "DISTRICT" in gdf.columns:
    gdf = gdf[gdf["DISTRICT"].astype(str).str.contains("Lahore", case=False, na=False)]

if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

if "uc_id" not in gdf.columns:
    print("Adding 'uc_id' column as unique identifier.")
    gdf["uc_id"] = range(1, len(gdf) + 1)

ucs = geemap.geopandas_to_ee(gdf) # <-- ee.FeatureCollection in memory

Adding 'uc_id' column as unique identifier.


Use cell below if extracting from uploaded shape file on GEE directly

In [3]:
uc_asset = "projects/ee-ahmedabclr35/assets/Lahore_union_Council_Boundries"
ucs = ee.FeatureCollection(uc_asset)

## Start and end date

In [ ]:
start = "2024-01-01"  
end = "2024-08-30"

## 1) NDVI 

In [4]:
from ndvi import compute_uc_ndvi

results = compute_uc_ndvi(ucs, start, end)


Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/NDVI.geojson
GeoJSON saved to NDVI.geojson
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/NDVI.geojson
GeoJSON saved to NDVI.geojson


In [ ]:
import numpy as np, geopandas as gpd, folium
uc_gdf = gpd.read_file("NDVI.geojson")
val_col = "NDVI" if "NDVI" in uc_gdf.columns else ("mean" if "mean" in uc_gdf.columns else None)
if val_col is None:
    raise ValueError("No NDVI column found in NDVI.geojson; expected 'NDVI' or 'mean'.")
uc_gdf = uc_gdf.dropna(subset=[val_col])
if uc_gdf.crs and uc_gdf.crs.to_epsg() != 4326:
    uc_gdf = uc_gdf.to_crs(4326)
center = [uc_gdf.geometry.centroid.y.mean(), uc_gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
vmin, vmax = float(uc_gdf[val_col].min()), float(uc_gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(uc_gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))
folium.Choropleth(
    geo_data=uc_gdf, data=uc_gdf,
    columns=["UC", val_col], key_on="feature.properties.UC",
    fill_color="YlGn", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, nan_fill_opacity=0, legend_name="Mean NDVI",
).add_to(m)
folium.GeoJson(
    uc_gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(fields=["UC", val_col], aliases=["UC Name", "NDVI Mean"], localize=True)
).add_to(m)
m.save("NDVI_heatmap.html")
m

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_48532/3944028875.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


In [ ]:

# --- 2) Build NDVI from Sentinel-2 SR ---
def mask_s2_sr_clouds(img):
    # SCL classes to drop: 3=shadow, 8=cloud, 9=cirrus, 10=snow/ice, 11=saturated/defective
    scl = img.select("SCL")
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return img.updateMask(mask).copyProperties(img, img.propertyNames())

s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
        .filterDate(start, end)
        .filterBounds(ucs)
        .map(mask_s2_sr_clouds)
        .select(["B4","B8"])
        .median())

ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")

# (Optional) lock projection to native 10m
ndvi = ndvi.reproject(crs="EPSG:4326", scale=10)

# --- 3) Reduce to UC polygons ---
uc_stats = ndvi.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=10
)

# --- 4) Export locally (small/medium data) ---
# GeoJSON
geemap.ee_export_vector(uc_stats, filename="UC_NDVI.geojson")

# CSV
features = uc_stats.getInfo()['features']

# Flatten into list of dicts
rows = [f['properties'] for f in features]

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save locally
df.to_csv("UC_NDVI.csv", index=False)
print("Saved UC NDVI locally!")



### Plot the heatmap on Folium 

In [ ]:
import numpy as np, geopandas as gpd, folium
uc_gdf = gpd.read_file("UC_NDVI.geojson")
val_col = "NDVI" if "NDVI" in uc_gdf.columns else ("mean" if "mean" in uc_gdf.columns else None)
if val_col is None:
    raise ValueError("No NDVI column found in UC_NDVI.geojson; expected 'NDVI' or 'mean'.")
uc_gdf = uc_gdf.dropna(subset=[val_col])
if uc_gdf.crs and uc_gdf.crs.to_epsg() != 4326:
    uc_gdf = uc_gdf.to_crs(4326)
center = [uc_gdf.geometry.centroid.y.mean(), uc_gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
vmin, vmax = float(uc_gdf[val_col].min()), float(uc_gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(uc_gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))
folium.Choropleth(
    geo_data=uc_gdf, data=uc_gdf,
    columns=["UC", val_col], key_on="feature.properties.UC",
    fill_color="YlGn", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, nan_fill_opacity=0, legend_name="Mean NDVI",
).add_to(m)
folium.GeoJson(
    uc_gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(fields=["UC", val_col], aliases=["UC Name", "NDVI Mean"], localize=True)
).add_to(m)
m.save("UC_NDVI_heatmap.html")
m

DataSourceError: 'UC_NDVI.geojson' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

## 2) Nightlights

In [ ]:
# --- VIIRS Nightlights by UC (GeoJSON + CSV export) ---

# Prefer nightly lights stable, cloud-free product; fall back if empty
viirs_primary_id = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"   # stray-light corrected
viirs_fallback_id = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"    # fallback if above empty

def viirs_collection(col_id):
    return (ee.ImageCollection(col_id)
            .filterDate(start, end)
            .filterBounds(ucs)
            .select(["avg_rad"]))

col = viirs_collection(viirs_primary_id)
count = col.size().getInfo()
if count == 0:
    col = viirs_collection(viirs_fallback_id)
    print("Primary VIIRS collection empty for range; using fallback VCMCFG.")

# Aggregate to mean nightlights for the period
viirs_mean = col.mean().rename("avg_rad")

# Reduce to UC-level features (mean radiance)
nl_stats = viirs_mean.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=500
)

# Export locally
geemap.ee_export_vector(nl_stats, filename="UC_NL.geojson")

# Get all features into Python dict
features = nl_stats.getInfo()['features']

# Flatten into list of dicts
rows = [f['properties'] for f in features]

# Convert to DataFrame
df_nl = pd.DataFrame(rows)

# Save locally
df_nl.to_csv("UC_NL.csv", index=False)
print("Saved UC Nightlights locally!")


In [ ]:
import numpy as np, geopandas as gpd, folium
gdf = gpd.read_file("UC_NL.geojson")
val_col = "avg_rad" if "avg_rad" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
if val_col is None:
    raise ValueError("No nightlights column found in UC_NL.geojson; expected 'avg_rad' or 'mean'.")
gdf = gdf.dropna(subset=[val_col])
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
m_nl = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
vmin, vmax = float(gdf[val_col].min()), float(gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC", val_col], key_on="feature.properties.UC",
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, nan_fill_opacity=0, legend_name="VIIRS Nightlights Mean",
).add_to(m_nl)
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    tooltip=folium.GeoJsonTooltip(fields=["UC", val_col], aliases=["UC Name:", "Nightlights Mean:"], localize=True)
).add_to(m_nl)
m_nl.save("UC_NL_heatmap.html")

# Building Footprint + LST + Pollution Map

This section will guide you through visualizing building footprints, Land Surface Temperature (LST), and pollution data for each Union Council (UC). You can merge these datasets with your UC boundaries and display them as interactive choropleth maps using Folium, similar to the NDVI and Nightlights visualizations above.

**Workflow:**
1. Prepare building footprint, LST, and pollution datasets (GeoJSON/CSV).
2. Merge with UC boundaries on the UC name or ID.
3. Use Folium to render choropleth maps for each indicator.
4. Customize tooltips and legends for clear interpretation.

_Example fields to visualize:_
- **Building Footprint:** Total built-up area or % coverage per UC.
- **LST:** Mean or max temperature per UC.
- **Pollution:** PM2.5, NO2, or other metrics per UC.

_See previous cells for code templates to join and plot these datasets._

## LST (Land Surface Temperature)

In [ ]:
# --- MODIS Land Surface Temperature (LST) extraction for UCs ---
# Uses MODIS Terra LST Daily product (1km resolution) and exports per-UC mean LST,
# plus optional area and valid coverage diagnostics.

import ee, geemap, pandas as pd

modis_lst_id = "MODIS/061/MOD11A1"

# Keep only best-quality pixels (QC_Day == 0)
def mask_lst_quality(img):
    qc = img.select("QC_Day")
    mask = qc.eq(0)
    return img.updateMask(mask)

# Load and filter MODIS LST collection
lst_collection = (
    ee.ImageCollection(modis_lst_id)
    .filterDate(start, end)
    .filterBounds(ucs)
    .map(mask_lst_quality)
)

# Check collection size for diagnostics
collection_size = lst_collection.size().getInfo()
print(f"MODIS LST collection size for {start} to {end}: {collection_size} images")
if collection_size == 0:
    print("WARNING: No MODIS LST images found for this date range!")
    print("Try using historical dates (e.g., 2023 or 2024) for better data availability.")

# Convert Kelvin*scale to Celsius: LST_Day_1km units are Kelvin * 0.02
def to_celsius(img):
    lst_k = img.select("LST_Day_1km").multiply(0.02)
    lst_c = lst_k.subtract(273.15).rename("LST_C")
    return lst_c

# Composite and convert to Celsius
lst_mean = lst_collection.map(to_celsius).mean()

# Reduce to UC-level stats
uc_lst = lst_mean.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.mean(),
    scale=1000
)

# --- Enhanced export: include area and coverage diagnostics ---
# Compute valid pixel area per UC
masked_area = lst_mean.select("LST_C").mask().multiply(ee.Image.pixelArea())
uc_area_coverage = masked_area.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.sum().setOutputs(["valid_km2"]),
    scale=1000
)

# Export main LST results
geemap.ee_export_vector(uc_lst, filename="UC_LST.geojson")

# Export coverage diagnostics separately
geemap.ee_export_vector(uc_area_coverage, filename="UC_LST_coverage.geojson")

# Create CSV outputs
features = uc_lst.getInfo()['features']
rows = [f['properties'] for f in features]
df_lst = pd.DataFrame(rows)
if "mean" in df_lst.columns:
    df_lst = df_lst.rename(columns={"mean": "LST_C_mean"})
df_lst.to_csv("UC_LST.csv", index=False)

# Enhanced CSV with area diagnostics
coverage_features = uc_area_coverage.getInfo()['features']
coverage_rows = [f['properties'] for f in coverage_features]
df_coverage = pd.DataFrame(coverage_rows)

# Add UC area in km2
def add_area_km2(f):
    area_m2 = f.geometry().area(1)
    area_km2 = area_m2.divide(1e6)
    return f.set("Area_km2", area_km2)

uc_with_area = ucs.map(add_area_km2)
area_features = uc_with_area.getInfo()['features']
area_rows = [f['properties'] for f in area_features]
df_area = pd.DataFrame(area_rows)

# Merge all diagnostics
df_enhanced = df_lst.merge(df_area[["UC", "Area_km2"]], on="UC", how="left")
df_enhanced = df_enhanced.merge(df_coverage[["UC", "valid_km2"]], on="UC", how="left")
df_enhanced["valid_km2"] = df_enhanced["valid_km2"] / 1e6  # Convert m2 to km2
df_enhanced["coverage_pct"] = (df_enhanced["valid_km2"] / df_enhanced["Area_km2"]) * 100

df_enhanced.to_csv("UC_LST_with_area.csv", index=False)

print("Saved UC LST locally!")
# Show any UCs with very low coverage
low_coverage = df_enhanced[df_enhanced["coverage_pct"] < 10]
if not low_coverage.empty:
    print(f"UCs with <10% LST coverage: {list(low_coverage['UC'])}")

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/UC_LST.geojson
Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/GEE/UC_LST_coverage.geojson
Saved UC LST locally!
UCs with <10% LST coverage: ['Johar Town', 'Chandrai', 'Bhaseen', 'Bhaseen', 'Al-faisal Town', 'Green Town', 'Maryam Colony', 'Keer Kalan', 'Township Sector A', 'Township', 'Sittara Colony', 'Farid Colony', 'Ismail Nagar', 'Bostan Colony', 'Awan Town', 'Sikandar Block', 'Muslim Town', 'Kashmir Block', 'Gulshan-e-iqbal', 'Faisal Town', 'Garden Town', 'Model Town', 'Liaqatabad', 'Pindi Rajputan', 'Kot Lakhpat', 'Naseer Abad', 'Makkah Colony', 'Gulberg', 'Al-hamra', 'Race Course', 'Shadman', 'Rehman Pura', 'Tajpura', 'Ghaziabad', 'Mustafa Abad', 'Mian Meer', 'Zaman Park', 'Daras Baray Mian', 'Mughalpura', 'Mujahidabad', 'Nabipura', 'Fateh Garh', 'Fateh Garh', 'Fateh Garh', 'Fateh

### Plot on Folium

In [6]:
import geopandas as gpd, folium, numpy as np

# 1) Load UC LST results (from your EE export)
gdf = gpd.read_file("UC_LST.geojson")

# 2) Figure out which column holds LST (handle both cases)
val_col = "LST_C" if "LST_C" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
if val_col is None:
    raise ValueError("No LST column found. Expected 'LST_C' or 'mean' in UC_LST.geojson")

# 3) Drop NAs and ensure lat/lon CRS for Folium (EPSG:4326)
gdf = gdf.dropna(subset=[val_col])
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

# 4) Map center
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]

# 5) Create map
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# 6) Nice bins for the legend (quantiles fallback to linspace if constant)
vmin, vmax = float(gdf[val_col].min()), float(gdf[val_col].max())
if np.isfinite(vmin) and np.isfinite(vmax) and vmin != vmax:
    bins = list(np.quantile(gdf[val_col], [0, 0.2, 0.4, 0.6, 0.8, 1]))
else:
    bins = list(np.linspace(vmin, vmax if vmin != vmax else vmin + 1e-6, 6))

# 7) Choropleth
folium.Choropleth(
    geo_data=gdf,
    data=gdf,
    columns=["UC", val_col],                 # assumes a 'UC' name/id field exists
    key_on="feature.properties.UC",
    fill_color="YlOrRd",                     # good for temperatures
    fill_opacity=0.7,
    line_opacity=0.2,
    bins=bins,
    nan_fill_opacity=0,
    legend_name="Mean LST (°C)",
).add_to(m)

# 8) Clickable popups
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(
        fields=["UC", val_col],
        aliases=["UC Name", "Mean LST (°C)"],
        localize=True
    )
).add_to(m)

# 9) Save
m.save("UC_LST_heatmap2.html")
m

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_56175/147574955.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## LPPI (Land-Pollution Proxy Index LPPI)

In [ ]:
def mask_s2_sr_clouds(img):
    scl = img.select("SCL")
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return img.updateMask(mask).copyProperties(img, img.propertyNames())

s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
        .filterDate(start, end)
        .filterBounds(ucs)
        .map(mask_s2_sr_clouds)
        .median()
        .clip(ucs))

# --- 2) Indices: NDVI, NDBI, BSI (Sentinel-2 bands) ---
# NDVI
ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
# NDBI = (SWIR - NIR) / (SWIR + NIR)  -> (B11 - B8) / (B11 + B8)
ndbi = s2.normalizedDifference(['B11','B8']).rename('NDBI')
# BSI = ((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))
bsi = (s2.select('B11').add(s2.select('B4'))
         .subtract(s2.select('B8').add(s2.select('B2')))
         .divide(s2.select('B11').add(s2.select('B4'))
                 .add(s2.select('B8').add(s2.select('B2'))))
         .rename('BSI'))

stack = ndvi.addBands([ndbi, bsi]).clip(ucs)

# --- 3) Z-score normalize each band over the city extent ---
stats = stack.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=ucs.geometry(),
    scale=30,
    maxPixels=1e12
)
def z(img_band_name):
    mean = ee.Number(stats.get(f"{img_band_name}_mean"))
    std  = ee.Number(stats.get(f"{img_band_name}_stdDev"))
    return stack.select(img_band_name).subtract(mean).divide(std)

ndvi_z = z('NDVI')
ndbi_z = z('NDBI')
bsi_z  = z('BSI')

# --- 4) Land Pollution Proxy Index (LPPI) ---
# Heuristic weights: dumps/barren surfaces tend to have high BSI/NDBI and low NDVI.
# Tune weights if you ground-truth later.
lppi = (bsi_z.multiply(0.4)
        .add(ndbi_z.multiply(0.4))
        .add(ndvi_z.multiply(-0.2))
        .rename('LPPI'))

# --- (Optional) Proximity boosts: waste sites / rivers via OSM ---
# Requires geemap >= 0.30 and osmnx installed to run locally; comment out if not needed.
try:
    # Landfills / waste disposal
    osm_waste = geemap.osm_to_ee(
        north=32.6, south=31.0, east=75.2, west=73.5,  # bbox around Lahore; adjust if needed
        tags={'landuse': 'landfill', 'amenity': 'waste_disposal'}
    )
    waste_img = ee.Image().byte().paint(osm_waste, 1)
    # Distance in meters (fastDistanceTransform expects binary=1 where features are)
    waste_dist = waste_img.fastDistanceTransform(30, 'pixels').sqrt().multiply(30)
    waste_boost = waste_dist.multiply(-1).divide(500).exp()  # ~exp(-d/500m) in [~0,1]
    lppi = lppi.add(waste_boost.rename('WASTE_BOOST').multiply(0.2))
except Exception as _:
    pass  # skip if OSM not available in your env

# --- 5) Per-UC reduction ---
uc_lp = lppi.reduceRegions(collection=ucs, reducer=ee.Reducer.mean(), scale=30)

# --- 6) Export locally ---
geemap.ee_export_vector(uc_lp, filename="UC_LPPI.geojson")


In [ ]:
# Export the FeatureCollection to GeoJSON (already done in previous cell)
# geemap.ee_export_vector(uc_lp, filename="UC_LPPI.geojson")

# Read the exported GeoJSON and save as CSV
import pandas as pd
gdf = pd.read_json("UC_LPPI.geojson")
if "features" in gdf:
    rows = [f["properties"] for f in gdf["features"]]
    df = pd.DataFrame(rows)
    if "mean" in df.columns:
        df = df.rename(columns={"mean": "LPPI"})
    df.to_csv("UC_LPPI.csv", index=False)
    print("Saved UC LPPI locally!")
else:
    print("GeoJSON file not found or invalid format.")

In [ ]:
import geopandas as gpd, folium, numpy as np
gdf = gpd.read_file("UC_LPPI.geojson").dropna(subset=["mean"])
if gdf.crs and gdf.crs.to_epsg()!=4326:
    gdf = gdf.to_crs(4326)
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
bins = list(np.linspace(float(gdf["mean"].min()), float(gdf["mean"].max()), 7))
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC","mean"], key_on="feature.properties.UC",
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, legend_name="Land Pollution Proxy (LPPI, higher=worse)",
).add_to(m)
folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(fields=["UC","mean"], aliases=["UC","LPPI"])
).add_to(m)
m.save("UC_LPPI_heatmap.html")
m

## Building Footprint

In [ ]:
import ee, geemap

# -------------------------------
# OPTION A — Vector footprints (best): Google Open Buildings v3
# -------------------------------
# Notes:
# - Coverage includes South Asia (Pakistan). Property names: 'area_in_meters', 'confidence'. 
# - Filter by confidence to reduce false positives (0.75 is a good starting point).

BUILDING_MIN_CONF = 0.75   # tweak if needed
USE_EXACT_INTERSECTION = False  # True = clip buildings to UC before area (slower)

bld = (ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
         .filterBounds(ucs.geometry())
         .filter(ee.Filter.gte("confidence", BUILDING_MIN_CONF)))

def per_uc_building_metrics(feat):
    geom = feat.geometry()
    b = bld.filterBounds(geom)
    count = ee.Number(b.size())

    if USE_EXACT_INTERSECTION:
        # Exact footprint within the UC (slower)
        def clip_area(f):
            ia = f.geometry().intersection(geom, 1).area(1)
            return f.set({"clip_area_m2": ia})
        area_sum = ee.Number(b.map(clip_area).aggregate_sum("clip_area_m2"))
    else:
        # Fast: sum footprint areas from property (counts buildings that straddle boundaries fully)
        area_sum = ee.Number(b.aggregate_sum("area_in_meters"))

    uc_area = geom.area(1)
    density_per_km2 = count.divide(uc_area.divide(1e6))
    coverage_pct = area_sum.divide(uc_area).multiply(100)
    mean_area = ee.Algorithms.If(count.gt(0), area_sum.divide(count), 0)

    return (feat
            .set("bld_count", count)
            .set("bld_area_m2", area_sum)
            .set("bld_mean_m2", mean_area)
            .set("bld_density_km2", density_per_km2)
            .set("bld_coverage_pct", coverage_pct)
            .set("bld_conf_min", BUILDING_MIN_CONF)
            .set("bld_exact", int(USE_EXACT_INTERSECTION))
           )

uc_buildings = ucs.map(per_uc_building_metrics)

# Export (171 features → safe to sync export)
geemap.ee_export_vector(uc_buildings, filename="UC_Buildings_OpenBuildings.geojson")
# Also handy CSV:
geemap.ee_export_vector(uc_buildings, filename="UC_Buildings_OpenBuildings.csv")

print("Exported UC_Buildings_OpenBuildings.(geojson/csv)")

In [ ]:
import geopandas as gpd, folium, numpy as np

# Load exported building metrics
gdf = gpd.read_file("UC_Buildings_OpenBuildings.geojson").dropna(subset=["bld_coverage_pct"])
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
bins = list(np.linspace(float(gdf["bld_coverage_pct"].min()), float(gdf["bld_coverage_pct"].max()), 7))

m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=gdf, data=gdf,
    columns=["UC", "bld_coverage_pct"], key_on="feature.properties.UC",
    fill_color="YlGnBu", fill_opacity=0.7, line_opacity=0.2,
    bins=bins, legend_name="Building Coverage (%)",
).add_to(m)

folium.GeoJson(
    gdf,
    style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 0.3},
    popup=folium.GeoJsonPopup(
        fields=["UC", "bld_count", "bld_coverage_pct"],
        aliases=["UC", "Building Count", "Coverage (%)"]
    )
).add_to(m)

m.save("UC_Buildings_heatmap.html")
m

## High-rise Buildings

In [ ]:
# High-rise buildings per UC using Open Buildings Temporal v1 (latest year: 2023)
# - High-rise threshold: >= 30 m (tweak HR_MIN_M)
# - Presence threshold: >= 0.5 (tweak PRESENCE_MIN)
# - Outputs: CSV with hr_count_est, hr_density_km2, share %, etc.

import ee, geemap, pandas as pd
ee.Initialize()

# ----------------- config -----------------
YEAR = 2023          # latest available; dataset has 2016–2023 only
HR_MIN_M = 30        # "high-rise" threshold (≈10 floors)
PRESENCE_MIN = 0.5   # presence confidence threshold (uncalibrated)
SCALE_M = 4          # dataset's effective pixel size (m)
TILESCALE = 2        # bump to 4 if reductions run heavy

# `ucs` must already exist; if not, build it from your local boundaries before this cell.

# ----------------- data -------------------
col = (ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
         .filterBounds(ucs.geometry())
         .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01"))

img = col.mosaic()
height   = img.select("building_height")           # meters
presence = img.select("building_presence")         # 0..1 (uncalibrated)
frac     = img.select("building_fractional_count") # fractional count density (see EE docs)

# masks
build_mask = presence.gte(PRESENCE_MIN)
hr_mask    = build_mask.And(height.gte(HR_MIN_M))

# per-pixel metrics
pix_area = ee.Image.pixelArea()  # m²
count_all = frac.multiply(pix_area).updateMask(build_mask).rename("bld_count_est")
count_hr  = frac.multiply(pix_area).updateMask(hr_mask).rename("hr_count_est")
area_hr   = pix_area.updateMask(hr_mask).rename("hr_area_m2")

# stack and reduce (sum over each UC)
metrics = ee.Image.cat([count_all, count_hr, area_hr])
uc_sums = metrics.reduceRegions(
    collection=ucs,
    reducer=ee.Reducer.sum().repeat(3).setOutputs(["bld_count_est", "hr_count_est", "hr_area_m2"]),
    scale=SCALE_M,
    tileScale=TILESCALE
)

# add derived fields and metadata; strip geometry to keep payload tiny
def add_derived(f):
    f = ee.Feature(f)
    area_uc = f.geometry().area(1)  # m²
    total   = ee.Number(f.get("bld_count_est"))
    hr      = ee.Number(f.get("hr_count_est"))
    dens    = ee.Algorithms.If(area_uc.gt(0), hr.divide(area_uc.divide(1e6)), None)  # per km²
    share   = ee.Algorithms.If(total.gt(0), hr.divide(total).multiply(100), None)
    return ee.Feature(None, f.toDictionary().combine({
        "year": YEAR,
        "highrise_min_m": HR_MIN_M,
        "presence_min": PRESENCE_MIN,
        "hr_density_km2": dens,
        "hr_share_pct": share,
    }))

uc_out = uc_sums.map(add_derived)

# choose the props to keep (adjust ID fields to match your UCs)
keep = ["UC","uc_id","year","highrise_min_m","presence_min",
        "bld_count_est","hr_count_est","hr_share_pct","hr_area_m2","hr_density_km2"]
uc_out_slim = uc_out.map(lambda f: ee.Feature(None, ee.Feature(f).toDictionary(keep)))

# Export to CSV directly (robust for small collections)
csv_path = f"UC_Highrise_{YEAR}_min{HR_MIN_M}m.csv"
geemap.ee_export_vector(uc_out_slim, filename=csv_path)

# Sanity print: read the CSV and show top rows by hr_count_est
df = pd.read_csv(csv_path)
if "hr_count_est" in df.columns:
    print(df[["UC","hr_count_est","hr_density_km2","hr_share_pct"]].sort_values("hr_count_est", ascending=False).head(10))
print(f"Saved {csv_path}")

## Standardize Metrics and QC

In [ ]:
# --- Standardize metric columns and QC ---
import geopandas as gpd, pandas as pd, numpy as np, os
from pathlib import Path

def load_and_std(path, metric_name):
    if not Path(path).exists():
        return None, None
    gdf = gpd.read_file(path)
    # Decide value column
    if metric_name == "NDVI":
        val = "NDVI" if "NDVI" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
    elif metric_name == "NL":
        val = "avg_rad" if "avg_rad" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
    elif metric_name == "LST_C":
        val = "LST_C" if "LST_C" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
    elif metric_name == "LPPI":
        val = "LPPI" if "LPPI" in gdf.columns else ("mean" if "mean" in gdf.columns else None)
    else:
        val = None
    if val is None:
        print(f"Skip {path}: no column to standardize.")
        return None, None
    gdf = gdf.dropna(subset=[val])
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    # Rename to standard
    if val != metric_name:
        gdf = gdf.rename(columns={val: metric_name})
    # Dedupe UC if needed (keep first)
    if "UC" in gdf.columns:
        gdf = gdf.drop_duplicates(subset=["UC"])
    std_geo = Path(path).with_name(Path(path).stem + "_std.geojson")
    gdf.to_file(std_geo, driver="GeoJSON")
    # CSV
    df = pd.DataFrame(gdf.drop(columns="geometry"))
    std_csv = std_geo.with_suffix(".csv")
    df.to_csv(std_csv, index=False)
    print(f"Standardized {metric_name}: {len(gdf)} rows → {std_geo.name}, {std_csv.name}")
    return gdf, df

ndvi_gdf, ndvi_df = load_and_std("UC_NDVI.geojson", "NDVI")
nl_gdf, nl_df     = load_and_std("UC_NL.geojson", "NL")
lst_gdf, lst_df   = load_and_std("UC_LST.geojson", "LST_C")
lppi_gdf, lppi_df = load_and_std("UC_LPPI.geojson", "LPPI")

# Quick merged preview if at least two exist
dfs = [d for d in [ndvi_df, nl_df, lst_df, lppi_df] if d is not None]
if len(dfs) >= 2:
    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.merge(d[[c for c in d.columns if c != "geometry"]], on=["UC"], how="outer")
    print("Merged preview (first 5 rows):")
    print(merged.head())
else:
    print("Standardization done. Not enough datasets to preview merge.")